# 37. Django Web Framework — `django`
**Python Libraries Lab · Phase 3 · Web & Networking**

_The batteries-included framework: models, admin, views and tests._

⚙️ **Runs in Colab** after the setup cell below.

Django is a full web framework that comes with almost everything a real website needs. A database layer called the O R M, so you write Python classes instead of SQL. An automatic admin site to manage your data. User accounts, forms, security protections and a test client. Instagram and many other large sites started on Django.

**In this notebook:** You will create a ShopEasy project with a shop app, define a product model, create the database with migrations, add products, write a page and a JSON endpoint, test them with Django's test client, and run the real server with the admin site, inside Colab.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q django
import importlib.metadata as md
for pkg in ['django']:
    print(pkg, md.version(pkg))

## Step 1 · Create a project and an app
`startproject` makes the site; `startapp` makes a feature folder.

In [ ]:
!django-admin startproject shopsite
!cd shopsite && python manage.py startapp shop
!find shopsite -name "*.py" | sort | head -20

## Step 2 · Register the app and allow Colab's address
Add `shop` to `INSTALLED_APPS`. `ALLOWED_HOSTS = ['*']` is only for practice, never for a real site.

In [ ]:
from pathlib import Path
settings = Path("shopsite/shopsite/settings.py")
text = settings.read_text()
if '"shop"' not in text:
    text += '\nINSTALLED_APPS += ["shop"]\nALLOWED_HOSTS = ["*"]\nCSRF_TRUSTED_ORIGINS = ["https://*.googleusercontent.com"]\n'
    settings.write_text(text)
print(text[-160:])

## Step 3 · A model is a database table
Each attribute becomes a column. `__str__` controls how a product is shown in the admin.

In [ ]:
%%writefile shopsite/shop/models.py
from django.db import models

class Product(models.Model):
    name = models.CharField(max_length=100)
    price = models.DecimalField(max_digits=8, decimal_places=2)
    in_stock = models.BooleanField(default=True)
    created = models.DateTimeField(auto_now_add=True)

    def __str__(self):
        return f"{self.name} (Rs. {self.price})"

## Step 4 · Admin, views and URLs
One file each: register the model in the admin, write two views, map URLs to them.

In [ ]:
Path("shopsite/shop/admin.py").write_text("""from django.contrib import admin
from .models import Product
admin.site.register(Product)
""")
Path("shopsite/shop/views.py").write_text("""from django.http import HttpResponse, JsonResponse
from .models import Product

def product_list(request):
    rows = "".join(f"<li>{p.name}: Rs. {p.price}</li>" for p in Product.objects.filter(in_stock=True))
    return HttpResponse(f"<h1>ShopEasy products</h1><ul>{rows}</ul>")

def product_api(request):
    max_price = request.GET.get("max_price")
    qs = Product.objects.all()
    if max_price:
        qs = qs.filter(price__lte=max_price)
    return JsonResponse({"products": list(qs.values("id", "name", "price", "in_stock"))})
""")
Path("shopsite/shop/urls.py").write_text("""from django.urls import path
from . import views
urlpatterns = [path("", views.product_list, name="products"), path("api/products/", views.product_api, name="api")]
""")
Path("shopsite/shopsite/urls.py").write_text("""from django.contrib import admin
from django.urls import include, path
urlpatterns = [path("admin/", admin.site.urls), path("", include("shop.urls"))]
""")
print("Files written")

## Step 5 · Migrate and add data
`makemigrations` writes the table plan; `migrate` creates the tables. Then add products with the ORM.

In [ ]:
!cd shopsite && python manage.py makemigrations shop && python manage.py migrate -v 0
!cd shopsite && python manage.py shell -c "from shop.models import Product; Product.objects.all().delete(); [Product.objects.create(name=n, price=p, in_stock=s) for n, p, s in [('Steel Bottle', 499, True), ('Yoga Mat', 1299, True), ('Desk Lamp', 899, False), ('Notebook', 149, True)]]; print(Product.objects.count(), 'products'); print(list(Product.objects.filter(price__lt=900)))"

## Step 6 · Test with Django's test client
Tests run against a fresh temporary database, so they never touch your real data.

In [ ]:
Path("shopsite/shop/tests.py").write_text("""from django.test import TestCase
from .models import Product

class ShopTests(TestCase):
    def setUp(self):
        Product.objects.create(name="Steel Bottle", price=499)
        Product.objects.create(name="Old Lamp", price=899, in_stock=False)

    def test_page_lists_in_stock_only(self):
        r = self.client.get("/")
        self.assertEqual(r.status_code, 200)
        self.assertContains(r, "Steel Bottle")
        self.assertNotContains(r, "Old Lamp")

    def test_api_filters_by_price(self):
        data = self.client.get("/api/products/?max_price=500").json()
        self.assertEqual([p["name"] for p in data["products"]], ["Steel Bottle"])
""")
!cd shopsite && python manage.py test shop -v 2 2>&1 | tail -8

## Step 7 · Run the real server with the admin
Creates an admin user (username `admin`, password `shopeasy123`), starts the server and opens it.

In [ ]:
import os, subprocess, time
env = dict(os.environ, DJANGO_SUPERUSER_PASSWORD="shopeasy123")
subprocess.run("cd shopsite && python manage.py createsuperuser --noinput --username admin --email admin@example.com",
               shell=True, env=env, capture_output=True)
django_server = subprocess.Popen("cd shopsite && exec python manage.py runserver 0.0.0.0:8000", shell=True,
                                 stdout=open("django.log", "w"), stderr=subprocess.STDOUT)
time.sleep(4)
import urllib.request
print(urllib.request.urlopen("http://127.0.0.1:8000/api/products/").read().decode()[:200])
def open_in_browser(port, path="/"):
    try:
        from google.colab import output
        output.serve_kernel_port_as_window(port, path=path)
    except ImportError:
        print(f"Not in Colab: open http://localhost:{port}{path} in your browser")
open_in_browser(8000)          # try /admin/ too: log in as admin / shopeasy123

## Step 8 · Stop the server
Stop background servers when you are done.

In [ ]:
django_server.terminate()
print("Django server stopped")

## ✍️ Your turn
Use the Django shell command to count the products **cheaper than 1000** and store the printed number (as an int) in `answer`. Hint: capture the output with `out = !cd shopsite && python manage.py shell -c "..."` then `int(out[-1])`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    return answer == 3, f"Expected 3, got {answer}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
out = !cd shopsite && python manage.py shell -c "from shop.models import Product; print(Product.objects.filter(price__lt=1000).count())"
answer = int(out[-1])

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **Project / app**
- **Model**
- **Migration**
- **View**
- **URLconf**
- **Admin**
- **Test client**

**Next:** 38. Flask Micro-framework (`flask`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/38_flask.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install django
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.